# UK Energy Research Centre Hackathon 2026
## Track 03: Future Electricity Systems Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0ladayo/UK-Energy-Research-Centre-Hackathon/blob/main/notebooks/track_03_future_electricity_systems_starter.ipynb)

### Challenge Focus:
Analyze national grid dynamics, dynamic consumer flexibility, renewable generation mix, and grid decarbonization.
- **Key Themes**: Real-time pricing incentives (Octopus Agile half-hourly tariffs), national carbon intensity (forecast vs. actual), generation mix by fuel type (wind, solar, gas, nuclear, interconnectors), and regional electricity demand.
- **Data Dictionary**: Refer to `DATASET_DICTIONARY.md` for full schema, field descriptions, and units.


### 1. Environment Setup & Data Loading
If running in **Google Colab**, this cell automatically clones the repository and navigates into the workspace directory.


In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    print("Running in Google Colab environment...")
    !git clone https://github.com/0ladayo/UK-Energy-Research-Centre-Hackathon.git
    %cd UK-Energy-Research-Centre-Hackathon
else:
    if os.path.exists("Future Electricity Systems"):
        print("Running locally from repository root.")
    elif os.path.exists("../Future Electricity Systems"):
        os.chdir("..")
        print(f"Changed working directory to: {os.getcwd()}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")


### 2. Loading Track 03 Datasets
We load:
1. **Octopus Agile Dynamic Tariffs 2024**: Full year 30-minute consumer electricity rates (p/kWh).
2. **National Grid Carbon Intensity 2024**: 30-minute GB carbon intensity (gCO2/kWh).
3. **Elexon BMRS Generation by Fuel Type 2024**: Generation mix (Wind, Solar, CCGT Gas, Nuclear, etc.).
4. **Subnational Electricity Consumption**: Regional and local authority annual electricity usage.


In [ ]:
data_dir = "Future Electricity Systems"

# 1. Octopus Agile Tariffs
tariffs_path = os.path.join(data_dir, "octopus_agile_tariffs_30min_2024_full.csv")
df_tariffs = pd.read_csv(tariffs_path)
df_tariffs["valid_from"] = pd.to_datetime(df_tariffs["valid_from"])
df_tariffs["valid_to"] = pd.to_datetime(df_tariffs["valid_to"])
print("Agile Tariffs Shape:", df_tariffs.shape)
display(df_tariffs.head(3))


In [ ]:
# 2. GB Carbon Intensity
carbon_path = os.path.join(data_dir, "gb_carbon_intensity_30min_2024.csv")
df_carbon = pd.read_csv(carbon_path)
df_carbon["from"] = pd.to_datetime(df_carbon["from"])
df_carbon["to"] = pd.to_datetime(df_carbon["to"])
print("Carbon Intensity Shape:", df_carbon.shape)
display(df_carbon.head(3))


In [ ]:
# 3. GB Generation Mix (BMRS Elexon)
gen_path = os.path.join(data_dir, "gb_generation_mix_30min_2024.csv")
df_gen = pd.read_csv(gen_path)
df_gen["startTime"] = pd.to_datetime(df_gen["startTime"])
print("Generation Mix Shape:", df_gen.shape)
print("Fuel Types Present:", df_gen["fuelType"].unique().tolist())
display(df_gen.head(3))


### 3. Exploratory Analysis & Grid Dynamics
#### Analysis 1: Negative Price Events & Price Spikes in Octopus Agile 2024


In [ ]:
# Identify negative price events (plunge pricing where users get paid to consume)
negative_prices = df_tariffs[df_tariffs["value_inc_vat"] < 0]
print(f"Total half-hour settlement periods with negative tariff in 2024: {len(negative_prices)}")
print(f"Lowest price recorded: {df_tariffs['value_inc_vat'].min():.2f} p/kWh")
print(f"Highest price recorded: {df_tariffs['value_inc_vat'].max():.2f} p/kWh")

plt.figure(figsize=(12, 4))
plt.hist(df_tariffs["value_inc_vat"], bins=60, color="#0275d8", edgecolor="black")
plt.axvline(0, color="red", linestyle="--", label="0 p/kWh Threshold")
plt.title("Distribution of Octopus Agile Half-Hourly Prices (2024)")
plt.xlabel("Price (pence / kWh including VAT)")
plt.ylabel("Number of 30-min Intervals")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


#### Analysis 2: Correlation between Grid Carbon Intensity and Electricity Price


In [ ]:
# Align timestamps to evaluate tariff response to carbon intensity
# Resample or match on UTC timestamp
df_tariffs_indexed = df_tariffs.set_index("valid_from").sort_index()
df_carbon_indexed = df_carbon.set_index("from").sort_index()

merged_grid = pd.merge_asof(
    df_tariffs_indexed,
    df_carbon_indexed[["intensity_actual", "intensity_forecast", "intensity_index"]],
    left_index=True,
    right_index=True,
    tolerance=pd.Timedelta("15min")
).dropna(subset=["intensity_actual", "value_inc_vat"])

print("Merged Grid Alignment Shape:", merged_grid.shape)

plt.figure(figsize=(8, 6))
plt.scatter(merged_grid["intensity_actual"], merged_grid["value_inc_vat"], alpha=0.15, color="#5bc0de", s=10)
plt.title("Octopus Agile Price vs. GB Carbon Intensity (2024)")
plt.xlabel("Carbon Intensity (gCO2/kWh)")
plt.ylabel("Agile Tariff (p/kWh)")
plt.grid(True, alpha=0.3)
plt.show()


#### Analysis 3: Total Generation Share by Fuel Type


In [ ]:
total_by_fuel = df_gen.groupby("fuelType")["generation"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
total_by_fuel.plot(kind="bar", color="#5cb85c", edgecolor="black")
plt.title("Total Electricity Generation by Fuel Type (2024 - MWh)")
plt.xlabel("Fuel Type")
plt.ylabel("Total Generation (MWh)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.show()


### 4. Hackathon Starter Ideas & Challenge Questions
- **Battery Storage / Smart Charging Optimization**: Simulate an automated battery or EV smart charging algorithm that maximizes charging during negative/low price or low carbon hours and discharges during peak hours.
- **Forecasting & Anomaly Detection**: Build a model forecasting the probability of negative price intervals or extreme generation ramp-downs 24 hours in advance.
- **Renewable Curtailment Analysis**: Analyze instances where high wind generation coincides with grid congestion or price drops.
